In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

## BM25Retriever

LangChain 文档：[BM25 Retriever](https://python.langchain.com/v0.2/docs/integrations/retrievers/bm25/)

### 安装包

`BM25Retriever` 使用 rank_bm25 包。

In [2]:
# !pip install rank_bm25

In [3]:
# pip install rank_bm25 == 0.2.2

### 创建检索器（from_texts）

In [4]:
from langchain_community.retrievers import BM25Retriever

In [5]:
retriever = BM25Retriever.from_texts(["foo", "bar", "world", "hello", "foo bar"])

In [6]:
result = retriever.invoke("foo")
result

[Document(page_content='foo'),
 Document(page_content='foo bar'),
 Document(page_content='hello'),
 Document(page_content='world')]

In [7]:
retriever.k

4

#### 设置返回文档数

In [8]:
retriever = BM25Retriever.from_texts(["foo", "bar", "world", "hello", "foo bar"])
retriever.k = 2  # 设置返回文档数

In [9]:
result = retriever.invoke("foo")
result

[Document(page_content='foo'), Document(page_content='foo bar')]

### 创建检索器（from_documents）

In [10]:
from langchain_core.documents import Document

retriever = BM25Retriever.from_documents(
    [
        Document(page_content="foo"),
        Document(page_content="bar"),
        Document(page_content="world"),
        Document(page_content="hello"),
        Document(page_content="foo bar"),
    ]
)

In [11]:
result = retriever.invoke("foo")
result

[Document(page_content='foo'),
 Document(page_content='foo bar'),
 Document(page_content='hello'),
 Document(page_content='world')]

## EnsembleRetriever

In [13]:
doc_list_1 = [
    "我喜欢苹果",
    "我喜欢橙子",
    "苹果和橙子是水果",
]

doc_list_2 = [
    "你喜欢苹果",
    "你喜欢橙子",
]

In [14]:
from langchain_community.retrievers import BM25Retriever


# 初始化 BM25 retriever
bm25_retriever = BM25Retriever.from_texts(
    doc_list_1, metadatas=[{"source": "bm25"}] * len(doc_list_1)
)

bm25_retriever.k = 2

In [15]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


# 初始化 Faiss retriever
embedding = OpenAIEmbeddings()
faiss_vectorstore = FAISS.from_texts(
    doc_list_2, embedding, metadatas=[{"source": "faiss"}] * len(doc_list_2)
)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

In [16]:
from langchain.retrievers import EnsembleRetriever

# 初始化 ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

In [17]:
docs = ensemble_retriever.invoke("苹果")
docs

[Document(page_content='苹果和橙子是水果', metadata={'source': 'bm25'}),
 Document(page_content='你喜欢苹果', metadata={'source': 'faiss'}),
 Document(page_content='我喜欢橙子', metadata={'source': 'bm25'}),
 Document(page_content='你喜欢橙子', metadata={'source': 'faiss'})]

In [18]:
from langchain.retrievers import EnsembleRetriever

# 初始化 ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.2, 0.8]
)

In [19]:
docs = ensemble_retriever.invoke("苹果")
docs

[Document(page_content='你喜欢苹果', metadata={'source': 'faiss'}),
 Document(page_content='你喜欢橙子', metadata={'source': 'faiss'}),
 Document(page_content='苹果和橙子是水果', metadata={'source': 'bm25'}),
 Document(page_content='我喜欢橙子', metadata={'source': 'bm25'})]

### 在运行时配置检索器

In [20]:
from langchain_core.runnables import ConfigurableField

In [21]:
faiss_retriever = faiss_vectorstore.as_retriever(
    search_kwargs={"k": 2}
).configurable_fields(
    search_kwargs=ConfigurableField(
        id="search_kwargs_faiss",
        name="搜索参数",
        description="要使用的搜索参数",
    )
)

In [22]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

In [23]:
# 配置只从 FAISS 检索器返回一个源
config = {"configurable": {"search_kwargs_faiss": {"k": 1}}}
docs = ensemble_retriever.invoke("苹果", config=config)
docs

[Document(page_content='苹果和橙子是水果', metadata={'source': 'bm25'}),
 Document(page_content='你喜欢苹果', metadata={'source': 'faiss'}),
 Document(page_content='我喜欢橙子', metadata={'source': 'bm25'})]